In [ ]:
#from project import icu_query

/Users/jack/ucsf_courses_resources/winter 2026/datasci_223/final/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [2]:
from project import bucket_ecg_report_0

In [23]:
df = icu_query()


/Users/jack/ucsf_courses_resources/winter 2026/datasci_223/final/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [26]:
original_query = df # to compare against as we make edits to data

In [27]:
df.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,ecg_time,report_0
0,19942060,26995122,32178416,Neurology,Neurology,2161-01-11 16:09:00,2161-02-08 22:05:32,28.247593,2161-01-11 15:11:00,2161-02-10 13:41:00,...,40,168,186,268,484,43,-44,119,2161-01-12 08:49:00,Sinus tachycardia with sinus arrhythmia
1,17945971,25486527,36652195,Med/Surg,Med/Surg,2130-03-17 04:25:01,2130-03-18 15:04:34,1.444132,2130-03-16 19:53:00,2130-03-20 16:10:00,...,40,156,194,272,658,42,-3,64,2130-03-17 05:31:00,Sinus rhythm
2,13570398,29201840,30059691,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2182-10-19 17:28:12,2182-10-27 20:38:37,8.132234,2182-10-18 12:17:00,2182-10-28 14:10:00,...,40,136,202,298,636,18,0,83,2182-10-26 07:26:00,Sinus rhythm.
3,12506390,21301912,30062923,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2160-10-17 13:39:06,2160-10-18 14:19:00,1.027708,2160-10-16 10:30:00,2160-10-25 14:05:00,...,40,146,204,344,642,13,-37,154,2160-10-17 17:30:00,Sinus rhythm.
4,14178148,28718602,30126048,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2142-09-14 03:01:44,2142-09-16 19:13:11,2.674618,2142-09-13 22:53:00,2142-09-16 15:18:00,...,40,178,242,348,622,60,36,38,2142-09-14 20:13:00,Sinus rhythm


In [28]:
# need to bucket report_0...
# getting frequency of report_0 codes
original_ecg_report0_frequency = original_query['report_0'].value_counts()
original_ecg_report0_frequency.to_csv('sanity_outputs/original_ecg_report0_frequency.csv')


In [29]:
# sinus rhythm, sinus tachycardia, a.fib
df['report_0'] = df['report_0'].str.lower().str.removesuffix(".")
df = df[~df['report_0'].str.contains('warning')]
initial_drop = df['report_0'].value_counts()
initial_drop.to_csv('sanity_outputs/initial_drop.csv')
# handles: Sinus rhythm, Sinus tachycardia, Atrial fibrillation, Sinus bradycardia that had '.' suffix
# changes all to lower, drops records with 'warning' because those signals could harm our model
df = df[df['report_0'].notna()] # handles any NA values



In [30]:
# check if any NAs, there were none
print(original_query.shape)
print(df.shape)

(35474, 33)
(34830, 33)


In [31]:
# check mortality captured by these first four catgories ()
df_first_cats = df[df['report_0'].isin(['sinus rhythm', 'sinus tachycardia', 'atrial fibrillation', 'sinus bradycardia'])]
print(f"")
print(f"Mortalities captured by the first four report_0 buckets: {sum(df_first_cats['hospital_expire_flag'])} ({sum(df_first_cats['hospital_expire_flag']) / sum(data['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")


Mortalities captured by the first four report_0 buckets: 2061 (49.13%)
Total mortalities: 4195


In [ ]:
df['ecg_bucket'] = df['report_0'].apply(bucket_ecg_report_0)
# check mortality captured after bucket_ecg_report_0 function
df_cats = df[df['ecg_bucket'].isin(['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert'])]
print(f"Mortalities captured by the current buckets: {sum(df_cats['hospital_expire_flag'])} ({sum(df_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")
post_bucket_function = df['ecg_bucket'].value_counts()
post_bucket_function.to_csv('sanity_outputs/post_bucket_function.csv')

Mortalities captured by the current buckets: 3790 (90.35%)
Total mortalities: 4195
